# Press Fit (Interference Fit) Tolerance & Stress Relaxation Calculator
**Plastic Design Calculators — Notebook 4**

---

## Part 1: Theory and Governing Equations

### 1.1 The Viscoelastic Press Fit Problem

A press fit joins a rigid metal shaft into a plastic hub using radial interference pressure. Unlike metallic assemblies that retain full elastic energy, polymeric hubs continuously dissipate energy via **stress relaxation**: the contact pressure—and therefore the transmittable torque—decays logarithmically over decades of service.

### 1.2 Interference

The diametral interference $\delta$ is the dimensional overlap between shaft and hub bore:

$$\delta = D_1 - D$$

where $D_1$ is the shaft outer diameter and $D$ is the undisturbed hub bore diameter.

### 1.3 Contact Pressure (Modified Lamé)

Assuming the metal shaft is perfectly rigid (valid since $E_{\mathrm{steel}} \gg E_{\mathrm{plastic}}$), all deformation occurs in the plastic hub. The time-dependent contact pressure is:

$$\boxed{P(t) = \frac{\delta}{D_1} \cdot E_r(t) \cdot \frac{1}{A_{\mathrm{geo}} + \nu}}$$

where:
- $E_r(t)$ — relaxation modulus at service time $t$ [Pa]
- $\nu$ — Poisson's ratio of the plastic hub
- $A_{\mathrm{geo}}$ — geometry factor: $A_{\mathrm{geo}} = \dfrac{1 + (D_1/D_2)^2}{1 - (D_1/D_2)^2}$
- $D_2$ — hub outer diameter

### 1.4 Torque Capacity

Maximum transmittable torque via interfacial friction over the engagement length:

$$\boxed{M_t(t) = \frac{\pi}{2} D_1^2 \, L \, P(t) \, \mu}$$

where $L$ is the hub engagement length and $\mu$ is the static coefficient of friction.

### 1.5 Relaxation Modulus Interpolation

$E_r(t)$ is retrieved by log-linear interpolation of tabulated experimental data. The model assumes:

$$E_r(t) \approx E_0 \cdot \exp(-\lambda \ln t)  \quad \text{(log-linear decay)}$$

### Assumptions
- Metal shaft perfectly rigid ($E_{\mathrm{shaft}} \gg E_{\mathrm{hub}}$)
- Uniaxial hoop stress, thick-walled cylinder (Lamé)
- No thermal expansion mismatch modelled (constant temperature)
- Poisson's ratio constant over time

---
## Part 2: Variable Definitions and Unit Handling

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from utils.unit_registry import ureg, Q_, strip_units
from utils.material_db import RELAXATION_MODULUS, FRICTION_COEFFICIENTS

# ── Geometry ──────────────────────────────────────────────────────────────────
D1  = Q_(20.0, 'mm')    # shaft outer diameter (= hub bore after assembly)
D2  = Q_(34.0, 'mm')    # hub outer diameter
D   = Q_(19.8, 'mm')    # undisturbed hub bore diameter
L   = Q_(25.0, 'mm')    # hub engagement length

# ── Service parameters ────────────────────────────────────────────────────────
t_service  = Q_(87_600.0, 'hour')   # 10 years
M_required = Q_(15.0, 'N*m')        # minimum torque that must be transmitted

# ── Material ──────────────────────────────────────────────────────────────────
MAT_KEY = 'POM_unfilled'
mat     = RELAXATION_MODULUS[MAT_KEY]
nu      = mat['poisson']
mu      = FRICTION_COEFFICIENTS['POM_steel']

print(f"Material  : {mat['description']}")
print(f"ν         : {nu}")
print(f"μ (static): {mu}")
print(f"δ (diametral interference) : {strip_units((D1-D).to('mm')):.3f} mm")

---
## Part 3: Computation Engine

In [ ]:
# ── 3.1  Symbolic equations ───────────────────────────────────────────────────
delta_s, D1_s, D2_s, Er_s, nu_s, A_s = sp.symbols('delta D_1 D_2 E_r nu A', positive=True)
L_s, mu_s, P_s = sp.symbols('L mu P', positive=True)

A_geo_eq = (1 + (D1_s/D2_s)**2) / (1 - (D1_s/D2_s)**2)
P_eq     = (delta_s / D1_s) * Er_s * (1 / (A_s + nu_s))
Mt_eq    = (sp.pi / 2) * D1_s**2 * L_s * P_s * mu_s

print("Geometry factor A_geo =")
sp.pprint(A_geo_eq)
print()
print("Contact pressure P(t) =")
sp.pprint(P_eq)
print()
print("Torque capacity Mt(t) =")
sp.pprint(Mt_eq)

In [ ]:
def geometry_factor(D1_m, D2_m, **kwargs):
    """Lamé geometry factor for a thick-walled plastic hub.

    Args:
        D1_m (float): Hub bore (shaft) diameter [m].
        D2_m (float): Hub outer diameter [m].
        **kwargs: Reserved for elliptical hub cross-section corrections.

    Returns:
        float: Dimensionless geometry factor A_geo.
    """
    ratio2 = (D1_m / D2_m) ** 2
    return (1.0 + ratio2) / (1.0 - ratio2)


def relaxation_modulus_interpolated(t_hours, times_h, Er_Pa, **kwargs):
    """Log-linear interpolation of tabulated relaxation modulus E_r(t).

    Args:
        t_hours (float | numpy.ndarray): Query time(s) [hours].
        times_h (list[float]): Tabulated time points [hours].
        Er_Pa (list[float]): Tabulated relaxation modulus values [Pa].
        **kwargs: Reserved for WLF-shifted temperature-corrected modulus lookup.

    Returns:
        float | numpy.ndarray: Interpolated E_r [Pa].
    """
    log_t  = np.log10(np.asarray(t_hours))
    log_tbl = np.log10(times_h)
    log_Er  = np.log10(Er_Pa)
    f_interp = interp1d(log_tbl, log_Er, kind='linear',
                        bounds_error=False, fill_value=(log_Er[0], log_Er[-1]))
    return 10.0 ** f_interp(log_t)


def contact_pressure(delta_m, D1_m, Er_Pa_t, A_geo, nu, **kwargs):
    """Residual contact pressure after stress relaxation.

    Args:
        delta_m (float | ndarray): Diametral interference [m].
        D1_m (float): Shaft diameter [m].
        Er_Pa_t (float | ndarray): Relaxation modulus at time t [Pa].
        A_geo (float): Lamé geometry factor.
        nu (float): Poisson's ratio of hub material.
        **kwargs: Reserved for press-fit with thermal pre-stress offset.

    Returns:
        float | ndarray: Contact pressure P(t) [Pa].
    """
    return (delta_m / D1_m) * Er_Pa_t / (A_geo + nu)


def torque_capacity(D1_m, L_m, P_Pa, mu, **kwargs):
    """Maximum transmittable torque of the interference fit joint.

    Args:
        D1_m (float | ndarray): Shaft diameter [m].
        L_m (float): Engagement length [m].
        P_Pa (float | ndarray): Contact pressure [Pa].
        mu (float): Static coefficient of friction.
        **kwargs: Reserved for knurled shaft surface enhancement factors.

    Returns:
        float | ndarray: Torque capacity Mt [N·m].
    """
    return (np.pi / 2.0) * D1_m**2 * L_m * P_Pa * mu


print("Press-fit functions defined.")

In [ ]:
# ── 3.2  Numerical evaluation ─────────────────────────────────────────────────
D1_m  = strip_units(D1.to('meter'))
D2_m  = strip_units(D2.to('meter'))
D_m   = strip_units(D.to('meter'))
L_m   = strip_units(L.to('meter'))
delta_m = D1_m - D_m    # diametral interference [m]

A_geo = geometry_factor(D1_m, D2_m)

# Log-spaced time array: 0.01 h → t_service
t_service_h = strip_units(t_service.to('hour'))
t_arr_h = np.logspace(-2, np.log10(t_service_h), 400)

Er_arr  = relaxation_modulus_interpolated(t_arr_h, mat['times_h'], mat['Er_Pa'])
P_arr   = contact_pressure(delta_m, D1_m, Er_arr, A_geo, nu)
Mt_arr  = torque_capacity(D1_m, L_m, P_arr, mu)

Mt_req  = strip_units(M_required.to('N*m'))
t_end_h = strip_units(t_service.to('hour'))
Mt_final = Mt_arr[-1]

# Find crossover time if torque drops below requirement
below = Mt_arr < Mt_req
if np.any(below):
    cross_idx = np.argmax(below)
    t_cross_h = t_arr_h[cross_idx]
    print(f"⚠ Torque drops below {Mt_req:.1f} N·m at t ≈ {t_cross_h:,.0f} hours ({t_cross_h/8760:.1f} years)")
else:
    t_cross_h = None
    print(f"✓ Torque remains above {Mt_req:.1f} N·m throughout {t_end_h/8760:.0f}-year service life.")

print(f"\nA_geo              : {A_geo:.4f}")
print(f"δ (diametral)      : {delta_m*1e3:.3f} mm")
print(f"P at t=0.01 h      : {P_arr[0]/1e6:.2f} MPa")
print(f"P at t=t_service   : {P_arr[-1]/1e6:.2f} MPa")
print(f"Mt at t_service    : {Mt_final:.2f} N·m")

---
## Part 4: Data Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Relaxation modulus and contact pressure vs log-time ───────────────
ax1 = axes[0]
color_P  = 'steelblue'
color_Er = 'darkorange'

ax1_twin = ax1.twinx()
ax1.semilogx(t_arr_h, P_arr / 1e6, color=color_P, lw=2, label='Contact pressure P(t)')
ax1_twin.semilogx(t_arr_h, Er_arr / 1e6, color=color_Er, lw=2, ls='--', label='E_r(t)')

ax1.set_xlabel('Time [hours]')
ax1.set_ylabel('Contact Pressure P [MPa]', color=color_P)
ax1_twin.set_ylabel('Relaxation Modulus E_r [MPa]', color=color_Er)
ax1.set_title(f'Stress Relaxation\n({mat["description"]})')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
ax1.grid(True, which='both', alpha=0.3)

# ── Plot 2: Torque capacity vs log-time ───────────────────────────────────────
ax2 = axes[1]
ax2.semilogx(t_arr_h, Mt_arr, color='steelblue', lw=2, label='Mt(t)')
ax2.axhline(Mt_req, color='tomato', ls='--', lw=1.5, label=f'Required Mt = {Mt_req:.1f} N·m')
if t_cross_h is not None:
    ax2.axvline(t_cross_h, color='tomato', ls=':', lw=1.5,
                label=f'Crossover at {t_cross_h:,.0f} h')
ax2.axvline(t_end_h, color='purple', ls=':', lw=1,
            label=f'Service end ({t_end_h/8760:.0f} yr)')
ax2.set_xlabel('Time [hours]')
ax2.set_ylabel('Torque Capacity Mt [N·m]')
ax2.set_title(f'Torque Decay over Service Life\n(D₁={D1_m*1e3:.0f} mm, L={L_m*1e3:.0f} mm)')
ax2.legend(fontsize=8)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('04_press_fit_output.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 5: Design Rule Validation

In [ ]:
pass_torque   = Mt_final >= Mt_req
pass_pressure = P_arr[0] / 1e6 <= 30.0   # typical max allowable contact pressure for POM

def badge(passed):
    return "\033[92m  PASS  \033[0m" if passed else "\033[91m  FAIL  \033[0m"

print("═" * 65)
print("  DESIGN RULE VALIDATION — PRESS FIT")
print("═" * 65)
print(f"  Torque at end of service life:")
print(f"    Mt(t_service) = {Mt_final:.2f} N·m")
print(f"    Required      = {Mt_req:.2f} N·m")
print(f"    Result        : {badge(pass_torque)}")
print()
print(f"  Initial contact pressure (yield check):")
print(f"    P(t→0)        = {P_arr[0]/1e6:.2f} MPa")
print(f"    Limit         = 30.00 MPa  (typical POM yield stress)")
print(f"    Result        : {badge(pass_pressure)}")
print("═" * 65)
overall = pass_torque and pass_pressure
if overall:
    print("  ✓ OVERALL: DESIGN PASSES press fit criteria.")
else:
    print("  ✗ OVERALL: DESIGN FAILS — increase δ, L, or select GF material.")
print("═" * 65)